# ANKIT LLM v0.1 — Colab Tesla T4 runbook

**Model:** 97,435,392 parameters (decoder-only Transformer, RoPE + RMSNorm + SwiGLU, tied embeddings).
**Nothing is shrunk here** — this is the same architecture as `configs/model.yaml`.

### Rules this notebook follows
1. Existing project files, checkpoints and history are **not** deleted or overwritten.
2. `fp32` is never assumed to be the right dtype on a T4 — `auto` resolves to **fp16 + GradScaler** (Turing has no bf16 tensor cores).
3. **Benchmark first, training second.** The pilot cell is short (300 steps). The long run cell is gated and must be started by you, consciously.
4. Every checkpoint is mirrored to Google Drive with `.partial -> rename`, so a disconnect cannot corrupt the latest good file.

### Order
`GPU check → mount Drive → clone repo → deps → data → BENCHMARK → pilot → (approval) → long run → SFT → eval → chat`

Runtime: **Runtime → Change runtime type → Hardware accelerator: T4 GPU** before starting.


In [ ]:
!nvidia-smi
import torch, platform, shutil
print("py:", platform.python_version(), "| torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability(0)
    print("gpu:", p.name, "| cc:", f"{cap[0]}.{cap[1]}", "| vram GB:", round(p.total_memory/1e9, 2))
    print("bf16 accelerated:", cap[0] >= 8, "(False on T4 -> we use fp16 + GradScaler)")
du = shutil.disk_usage("/content")
print("disk free GB:", round(du.free/1e9, 1))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/ankit-llm"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/reports", exist_ok=True)
print("drive ok ->", DRIVE)

In [ ]:
import os
REPO = "https://github.com/pushparaj9749/tool.git"
BRANCH = "arena/4c3ac418-tool"
if not os.path.isdir("/content/ankit-llm"):
    !git clone --branch $BRANCH $REPO /content/ankit-llm
%cd /content/ankit-llm
!git log --oneline -3
!git status --short | head

## Dependencies

Colab already ships a CUDA-enabled PyTorch — **do not reinstall it** (that is the
classic way to break a Colab runtime). We install only the small extras.

In [ ]:
!pip -q install pyyaml tokenizers numpy pynvml psutil pytest
import torch, yaml, tokenizers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| tokenizers", tokenizers.__version__)

## Data and checkpoints

Two options — pick **one**.

* **A. Restore from Drive** if you already uploaded `ankit-artifacts.zip` (tokenizer + token shards + optional checkpoints). Fast, no downloads of datasets.
* **B. Rebuild from source** — regenerates tokenizer + shards from the licensed corpora (~5-8 min on Colab, download over GitHub).

The tokenizer must be the *same* vocabulary the model was trained with. Never mix a rebuilt tokenizer with an old checkpoint — the code will refuse if `vocab_size` disagrees.

In [ ]:
import os, shutil, zipfile
ZIP = f"{DRIVE}/ankit-artifacts.zip"
if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content/ankit-llm")
    print("restored from Drive:", ZIP)
else:
    print("no zip on Drive -> run the rebuild cell below instead")
!ls -la tokenizer/ankit_bpe 2>/dev/null | head -5
!ls -la data/processed/*.bin 2>/dev/null

In [ ]:
# OPTION B — rebuild data + tokenizer + shards (skip if Option A worked)
import os
if not os.path.exists("data/processed/pretrain_train.bin") or not os.path.exists("tokenizer/ankit_bpe/tokenizer.json"):
    !bash scripts/download_data.sh
    !python data/prepare_data.py
    !python tokenizer/train_tokenizer.py --config configs/tokenizer.yaml --out-dir tokenizer/ankit_bpe
    !python data/build_token_shards.py
    !python data/build_sft.py
    !python data/validate_data.py
else:
    print("artifacts already present — nothing to do")
!ls -lh data/processed/*.bin

## Phase 3 — GPU benchmark (run this before any training)

Measures the **real 97.4M architecture**: fp32 / fp16 / bf16 × batch 1,2,4,8 × seq 256,512,
with warm-up steps, median-of-N timing, `cuda.synchronize()`, peak VRAM, GPU utilisation,
system RAM and numerical stability (every loss + grad norm + fp16 scaler scale).

OOM configs are reported as `oom` — not skipped silently.

In [ ]:
!python benchmarks/bench_gpu.py --out reports/gpu_benchmark --warmup 2 --steps 6
import shutil
shutil.copy("reports/gpu_benchmark.md", f"{DRIVE}/reports/gpu_benchmark.md")
shutil.copy("reports/gpu_benchmark.json", f"{DRIVE}/reports/gpu_benchmark.json")
print(open("reports/gpu_benchmark.md").read())

### Read the benchmark before continuing

Pick the config you will train with:

* **Highest tokens/s** that keeps `loss_finite: True` and `status: ok`.
* Sanity check: fp16 should beat fp32 by a clear margin; bf16 on T4 will likely be *slower* (no tensor cores) — that is expected, not a bug.
* If `scaler_scale` collapses toward 1.0 or losses go `nan`, do not use that dtype.
* Peak VRAM must stay under ~13.5 GB so validation and checkpointing have headroom.

Then set `batch_size` / `grad_accum_steps` in `configs/pretraining_colab_pilot.yaml` accordingly.

## Phase 5 — pilot training (short, gated)

300 steps ≈ 2.46M tokens. This proves the GPU path end-to-end: finite loss, decreasing
loss, validation, checkpoint, resume. It is **not** a release run.

In [ ]:
!python training/pretrain.py --config configs/pretraining_colab_pilot.yaml --run-id t4pilot --time-budget-sec 1800
!ls -lh checkpoints/t4-pilot | head
!cat checkpoints/t4-pilot/latest.json

### Verify the pilot honestly

1. `loss` trend is downward and every logged loss is finite.
2. A `val_loss` line exists (validation ran).
3. `latest.json` has a `sha256` and the mirrored copy exists on Drive.
4. **Resume test:** run the next cell — it must print `resumed from ... at step N`.

In [ ]:
# Resume test: continues the pilot from the last checkpoint for 20 more steps
!python training/pretrain.py --config configs/pretraining_colab_pilot.yaml --run-id t4pilot-resume --max-steps 320 --resume auto
!ls -lh /content/drive/MyDrive/ankit-llm/checkpoints/t4-pilot | tail -5

## ⛔ Stop here and report

Post the benchmark table + pilot result, then **wait for explicit approval** before the long run.
The long run below is ~2B tokens (~19-28 GPU-hours on a T4) and will span multiple Colab sessions.

## Full pretraining (only after approval, resumable across sessions)

Because the LR schedule is anchored to `max_steps`, stopping and resuming keeps you on the
same curve. Each session: run this cell until Colab dies, then start a new session, re-run
mount + clone + **restore from Drive**, and run it again.

Before each new session, restore the latest checkpoint from Drive:
```python
import shutil, os
os.makedirs("checkpoints/t4-full", exist_ok=True)
d = "/content/drive/MyDrive/ankit-llm/checkpoints/t4-full"
for f in os.listdir(d):
    if f.endswith((".pt", ".json")):
        shutil.copy(os.path.join(d, f), os.path.join("checkpoints/t4-full", f))
print("restored:", sorted(os.listdir("checkpoints/t4-full")))
```

In [ ]:
# GATED: full 100M pretraining (~2B tokens). Do not run without approval.
# RUN_FULL = False by default so an accidental "Run all" cannot start it.
RUN_FULL = False
if RUN_FULL:
    import shutil, os
    os.makedirs("checkpoints/t4-full", exist_ok=True)
    d = "/content/drive/MyDrive/ankit-llm/checkpoints/t4-full"
    if os.path.isdir(d):
        for f in os.listdir(d):
            if f.endswith((".pt", ".json")):
                shutil.copy(os.path.join(d, f), os.path.join("checkpoints/t4-full", f))
    !python training/pretrain.py --config configs/pretraining_colab_t4.yaml --resume auto --time-budget-sec 40000
else:
    print("Full run not started (set RUN_FULL = True when approved).")

## Instruction tuning (after pretraining has a usable checkpoint)

The SFT set has 500+ examples across Hindi / English / Hinglish and the three personality
modes (professional, friendly, adaptive). Loss is masked to assistant tokens only.

In [ ]:
!python training/finetune.py --config configs/finetuning_colab_t4.yaml --run-id t4sft
import shutil
shutil.copy("checkpoints/t4-sft/final.pt", f"{DRIVE}/checkpoints/t4-sft-final.pt")

## Evaluation and chat

In [ ]:
CKPT = "checkpoints/t4-sft/final.pt"
!python evaluation/evaluate_lm.py --checkpoint $CKPT
!python evaluation/evaluate_chat.py --checkpoint $CKPT
!python evaluation/evaluate_languages.py --checkpoint $CKPT
!python evaluation/evaluate_personality.py --checkpoint $CKPT
!python evaluation/collect_examples.py --checkpoint $CKPT
!python evaluation/report_summary.py
!cat reports/RELEASE_REPORT.md | head -60

In [ ]:
from inference.conversation import ChatEngine
eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
print("params:", eng.model.num_parameters())
for mode in ["professional", "friendly", "adaptive"]:
    print(f"\n[{mode}]", eng.chat_once([{"role":"user","content":"नमस्ते अंकित, तुम कौन हो?"}], mode=mode))
print("\n[hinglish]", eng.chat_once([{"role":"user","content":"Bhai, tokenizer kya hota hai?"}], mode="adaptive"))

In [ ]:
# Interactive loop (run manually, not with Run All)
# from inference.conversation import ChatEngine
# eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
# conv = eng.new_conversation("adaptive")
# while True:
#     q = input("you > ")
#     if q in ("/quit", "exit"): break
#     print("ankit>", eng.reply(conv, q))